In [ ]:
# ═══════════════════════════════════════════════════════
#  CELL 1 — GPU CHECK + DRIVE MOUNT
#  Run first. If GPU is missing, stop here and change runtime.
# ═══════════════════════════════════════════════════════
import os, sys, subprocess
from pathlib import Path
import torch
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU: {name}  ({vram:.1f} GB VRAM)")
else:
    raise RuntimeError(
        "\n✗ No GPU detected.\n"
        "Runtime → Change runtime type → Hardware accelerator → T4 GPU\n"
        "Then re-run this cell."
    )

print("✓ Drive mounted.")


In [ ]:
# ═══════════════════════════════════════════════════════
#  CELL 2 — INSTALL ALL DEPENDENCIES
#  Safe to re-run. Takes ~2 minutes on first run.
# ═══════════════════════════════════════════════════════
import sys, subprocess

packages = [
    "torch", "numpy", "PyYAML", "pytest", "tqdm",
    "sympy", "scipy", "psutil", "networkx",
    "fastapi", "uvicorn", "httpx",
    "sentence-transformers",
    "transformers", "accelerate",
]

print("Installing standard packages...")
for pkg in packages:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    icon = "✓" if r.returncode == 0 else "⚠"
    print(f"  {icon} {pkg}")

print("\nInstalling Muon optimizer (from GitHub)...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/KellerJordan/Muon"],
    capture_output=True, text=True
)
if r.returncode == 0:
    print("  ✓ muon")
else:
    print("  ⚠ muon install failed — AdamW fallback will be used automatically")

print("\n✓ All dependencies installed.")


In [ ]:
# ═══════════════════════════════════════════════════════
#  CELL 3 — CLONE OR UPDATE AN-RA
#  Clones fresh if not present. Pulls latest if already there.
# ═══════════════════════════════════════════════════════
import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"

if not REPO.exists():
    print("Cloning An-Ra (first time)...")
    r = subprocess.run(
        ["git", "clone", GITHUB_URL, str(REPO)],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f"Clone failed:\n{r.stderr}")
    print("✓ Cloned.")
else:
    r = subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only"],
        capture_output=True, text=True
    )
    print(f"✓ {r.stdout.strip() or 'Already up to date.'}")

os.environ["ANRA_REPO_ROOT"] = str(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from anra_paths import inject_all_paths, ensure_dirs
inject_all_paths()
ensure_dirs()
print(f"✓ Paths injected. Working dir: {REPO}")


In [ ]:
# ═══════════════════════════════════════════════════════
#  CELL 4 — RESTORE CHECKPOINTS + TOKENIZER FROM DRIVE
#  Must run before Cell 6. Restores your previous progress.
# ═══════════════════════════════════════════════════════
import os, sys, shutil, pickle, subprocess
from pathlib import Path
from google.colab import drive

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive', force_remount=False)
if not REPO.exists():
    subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from anra_paths import inject_all_paths, ensure_dirs, TOKENIZER_DIR, TRAINING_DATA_DIR, DRIVE_CHECKPOINTS
inject_all_paths()
ensure_dirs()

DRIVE = Path("/content/drive/MyDrive/AnRa")
print("=== Restoring An-Ra artifacts from Drive ===\n")

for name in ["anra_brain.pt", "anra_brain_identity.pt", "anra_ouroboros.pt"]:
    src = DRIVE / "checkpoints" / name
    dst = REPO / name
    if src.exists() and src.stat().st_size > 10_000:
        shutil.copy2(src, dst)
        print(f"  ✓ {name} restored ({dst.stat().st_size / 1e6:.1f} MB) → will RESUME training")
    else:
        print(f"  ○ {name}: not on Drive → fresh start")

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
tok_local = TOKENIZER_DIR / "tokenizer.pkl"
tok_drive = DRIVE / "tokenizer.pkl"

if tok_drive.exists() and tok_drive.stat().st_size > 100:
    shutil.copy2(tok_drive, tok_local)
    with open(tok_local, "rb") as f:
        _t = pickle.load(f)
    print(f"\n  ✓ tokenizer.pkl restored from Drive (vocab_size={_t.vocab_size})")
else:
    print("\n  ○ tokenizer.pkl not on Drive — build_brain.py will auto-create it at training start")

ds_local = TRAINING_DATA_DIR / "anra_dataset_v6_1.txt"
ds_drive = DRIVE / "anra_dataset_v6_1.txt"

if ds_local.exists():
    print(f"\n  ✓ Dataset: {ds_local} ({ds_local.stat().st_size / 1e6:.2f} MB)")
elif ds_drive.exists():
    TRAINING_DATA_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ds_drive, ds_local)
    print(f"\n  ✓ Dataset restored from Drive: {ds_local}")
else:
    print(
        f"\n  ✗ DATASET NOT FOUND.\n"
        f"    Upload 'anra_dataset_v6_1.txt' to Google Drive at:\n"
        f"      MyDrive/AnRa/anra_dataset_v6_1.txt\n"
        f"    Then re-run this cell.\n"
        f"    Training CANNOT start without the dataset."
    )

print("\n✓ Artifact restoration complete.")


In [ ]:
# ═══════════════════════════════════════════════════════
#  CELL 5 — SYSTEM HEALTH CHECK
#  Verify every module before training starts.
# ═══════════════════════════════════════════════════════
import os, sys, importlib, pickle, subprocess, torch
from pathlib import Path
from google.colab import drive

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive', force_remount=False)
if not REPO.exists():
    subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from anra_paths import inject_all_paths, ensure_dirs, TRAINING_DATA_DIR, TOKENIZER_DIR
inject_all_paths()
ensure_dirs()

print("╔══════════════════════════════════════════════════════╗")
print("║           AN-RA FULL SYSTEM HEALTH CHECK            ║")
print("╚══════════════════════════════════════════════════════╝\n")

print("── Critical Files ──")
critical = {
    "anra_brain.py": REPO / "anra_brain.py",
    "anra_paths.py": REPO / "anra_paths.py",
    "build_brain.py": REPO / "scripts" / "build_brain.py",
    "train_unified.py": REPO / "training" / "train_unified.py",
    "Training dataset": TRAINING_DATA_DIR / "anra_dataset_v6_1.txt",
}
for label, path in critical.items():
    if path.exists():
        size = f"  ({path.stat().st_size / 1e6:.2f} MB)" if path.suffix in {".py", ".txt"} else ""
        print(f"  ✓ {label}{size}")
    else:
        print(f"  ✗ {label}: MISSING — {path}")

print("\n── Tokenizer ──")
tok_path = TOKENIZER_DIR / "tokenizer.pkl"
if tok_path.exists():
    with open(tok_path, "rb") as f:
        tok = pickle.load(f)
    print(f"  ✓ tokenizer.pkl  vocab_size={tok.vocab_size}")
else:
    print("  ○ tokenizer.pkl: will be auto-built at training start")

print("\n── Checkpoint ──")
ckpt = REPO / "anra_brain.pt"
if ckpt.exists():
    try:
        state = torch.load(ckpt, map_location="cpu", weights_only=False)
        step = state.get("global_step", "?") if isinstance(state, dict) else "?"
        loss = state.get("best_loss", "?") if isinstance(state, dict) else "?"
        print(f"  ✓ anra_brain.pt  step={step}  best_loss={loss}")
    except Exception as e:
        print(f"  ⚠ anra_brain.pt: corrupted ({e}) — will train fresh")
else:
    print("  ○ anra_brain.pt: not found — will train from scratch")

print("\n── Module Health ──")
all_modules = {
    "TurboQuant         ": "turboquant",
    "Identity    (45N)  ": "identity_injector",
    "Ouroboros   (45O)  ": "ouroboros_numpy",
    "Ghost Mem   (45P)  ": "ghost_memory",
    "Symbolic    (45Q)  ": "symbolic_bridge",
    "Sovereignty (45R)  ": "sovereignty_bridge",
    "Memory      (45J)  ": "memory_manager",
    "Agent Loop  (45K)  ": "agent_main",
    "Self-Impr   (45L)  ": "improve",
    "Master Sys  (45M)  ": "system",
}
for label, mod_name in all_modules.items():
    try:
        mod = importlib.import_module(mod_name)
        fn = getattr(mod, "health_check", None)
        if callable(fn):
            result = fn()
            status = result.get("status", "ok") if isinstance(result, dict) else "ok"
            detail = result.get("reason", "") if isinstance(result, dict) and status != "ok" else ""
            icon = "✓" if status == "ok" else "⚠"
            suffix = f" — {detail}" if detail else ""
            print(f"  {icon} {label}: {status}{suffix}")
        else:
            print(f"  ✓ {label}: importable")
    except Exception as e:
        print(f"  ⚠ {label}: {type(e).__name__}: {e}")

print("\n── GPU ──")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  ✓ {torch.cuda.get_device_name(0)}  {used:.1f}/{total:.1f} GB")
else:
    print("  ✗ GPU not available — go to Runtime → Change runtime type → T4 GPU")

print("\n✓ Health check complete. If all critical files are present, proceed to Cell 6.")


In [ ]:
# ═══════════════════════════════════════════════════════
#  CELL 6 — TRAIN AN-RA
#  30-minute session. Auto-resumes from checkpoint.
#  Saves ONLY at the end — no interruptions during training.
# ═══════════════════════════════════════════════════════
import os, sys, subprocess, json
from pathlib import Path
from google.colab import drive

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive', force_remount=False)
if not REPO.exists():
    subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from anra_paths import inject_all_paths
inject_all_paths()

print("╔══════════════════════════════════════════════════════╗")
print("║              AN-RA TRAINING SESSION                 ║")
print("║  30 min · auto-resume · saves ONCE at end to Drive  ║")
print("╚══════════════════════════════════════════════════════╝\n")

cmd = [
    sys.executable, "-m", "training.train_unified",
    "--mode", "train",
    "--session_minutes", "30",
    "--checkpoint_path", "anra_brain.pt",
    "--block_size", "256",
    "--batch_size", "64",
    "--ouroboros_steps", "5000",
]

print("Command:", " ".join(cmd))
print("─" * 58)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(REPO),
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

print("─" * 58)

if proc.returncode == 0:
    print("\n✓ Training complete. Checkpoint saved to Drive.")
else:
    print(f"\n⚠ Training exited with code {proc.returncode}. Check output above.")

for rname in ["unified_training_report.json", "session_train_metrics.json"]:
    rp = REPO / "output" / rname
    if rp.exists():
        try:
            r = json.loads(rp.read_text())
            print(f"\n── {rname} ──")
            for k, v in r.items():
                if k == "stages":
                    print(f"  stages: {v}")
                else:
                    print(f"  {k}: {v}")
        except Exception:
            pass

drive_ckpt = Path("/content/drive/MyDrive/AnRa/checkpoints/anra_brain.pt")
if drive_ckpt.exists():
    print(f"\n✓ Drive backup: {drive_ckpt.stat().st_size / 1e6:.1f} MB")
    print("  Come back anytime — Cell 6 will resume exactly where this stopped.")
else:
    print(f"\n⚠ Drive backup not found at expected path: {drive_ckpt}")


In [ ]:
# ═══════════════════════════════════════════════════════
#  CELL 7 — POST-TRAINING VERIFICATION
#  Run after Cell 6 to confirm everything saved correctly.
# ═══════════════════════════════════════════════════════
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive', force_remount=False)
if not REPO.exists():
    subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from anra_paths import inject_all_paths
inject_all_paths()

print("═══ Post-Training Verification ═══\n")

r = subprocess.run(
    [sys.executable, "scripts/verify_structure.py"],
    capture_output=True, text=True, cwd=str(REPO)
)
print("── Structure ──")
print(r.stdout.strip())

r = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q", "--tb=short", "--no-header"],
    capture_output=True, text=True, cwd=str(REPO)
)
print("\n── Tests ──")
out = (r.stdout + r.stderr).strip()
print(out[-3000:] if len(out) > 3000 else out)

print("\n── Drive Backup ──")
drive = Path("/content/drive/MyDrive/AnRa")
for name in [
    "checkpoints/anra_brain.pt",
    "checkpoints/anra_brain_identity.pt",
    "tokenizer.pkl",
    "anra_dataset_v6_1.txt",
]:
    p = drive / name
    if p.exists():
        print(f"  ✓ {name}  ({p.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"  ○ {name}: not on Drive")

print("\n✓ Done.")
